# Max Activating Dataset Examples (MADE)

In [ ]:
import torch as t
import sklearn
import transformer_lens

from transformer_lens import ( 
    HookedTransformer, HookedTransformerConfig, ActivationCache, utilities
)

from datasets import load_dataset

t.manual_seed(42)

device = 'cpu'

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm
from IPython.display import HTML, display


/Users/ravisaulog/projects/mechinterp/minis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = HookedTransformer.from_pretrained('gpt2-small').to(device)
ds = load_dataset('NeelNanda/pile-10k')
sample_text = list(ds['train'][:50]['text'])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7216.34it/s]


Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu


In [4]:
sample_corpus= " ".join(sample_text)

In [5]:
BATCH_SIZE = 256

tokens = t.tensor(model.tokenizer.encode(sample_corpus))
batched_tokens = t.split(tokens, split_size_or_sections=BATCH_SIZE)

We will look at three random neurons in layer `6` of the model. `[1145, 466, 214]`

In [6]:
neuron_idx = [1145, 466, 214]

We will run the model on this corpus of text.

In [27]:
acts = [] # batch, seq, 3

with t.inference_mode():
    for i, batch in enumerate(tqdm(batched_tokens)) :

        batched = batch.unsqueeze(0)
        logits, cache = model.run_with_cache(batched, return_cache_object=True, names_filter=['blocks.6.mlp.hook_post'])
        neuron_acts = cache['blocks.6.mlp.hook_post'][..., neuron_idx].squeeze(0)# shape 1, 
        acts.append(neuron_acts)
        del cache 

    acts = t.cat(acts, dim=0).permute(1, 0)

100%|██████████| 139/139 [00:21<00:00,  6.50it/s]


In [31]:
acts.shape

torch.Size([3, 35394])

In [32]:
values, indices = acts.topk(k=15, dim=1)

### Token Inspection (Neuron 1) - 1145

In [ ]:
def grab_context(tokens, idx):
    return tokens[idx-10:idx+5]

def show(str_tokens, peak=10):
    parts = []
    for i, t in enumerate(str_tokens):
        t_safe = t.replace('<', '&lt;').replace('>', '&gt;').replace('\n', '⏎')
        if i == peak:
            parts.append(f'<span style="background:#ffeb3b;color:black;font-weight:bold">{t_safe}</span>')
        else:
            parts.append(t_safe)
    display(HTML(''.join(parts)))

In [52]:
for idx in indices[0]:
    ctx = grab_context(tokens=tokens, idx=idx)
    y = model.to_str_tokens(ctx)
    show(y)

This neuron seems to fire in two instances : when a word starts with `t`, or when it is `lo`. The other examples, and the fact that there are also two distinct instances of it firing, mean that this neuron is highly polysemantic.

We can verify by feeding some synthetic text.

### Token Inspection (Neuron 2)

In [53]:
for idx in indices[1]:
    ctx = grab_context(tokens=tokens, idx=idx)
    y = model.to_str_tokens(ctx)
    show(y)

### Token Inspection (Neuron 3)

In [54]:
for idx in indices[2]:
    ctx = grab_context(tokens=tokens, idx=idx)
    y = model.to_str_tokens(ctx)
    show(y)